# Groundwater Potential Mapping for Kano State, Nigeria

## GIS-Based Multi-Criteria Decision Analysis (MCDA)

This notebook develops a groundwater potential map for Kano State, Nigeria using GIS-based multi-criteria analysis. Environmental, topographic, hydrological, soil and landcover factors are prepared as thematic layers, reclassified to a common suitability scale, combined using a weighted overlay, and classified into five groundwater-potential zones.

### Objectives
1. Identify environmental and geological factors influencing groundwater occurrence.
2. Generate thematic layers representing groundwater controlling factors.
3. Apply GIS-based Multi Criteria Decision Analysis (MCDA).
4. Produce a groundwater potential map.
5. Classify groundwater potential into suitability zones.
6. Identify priority areas for groundwater development.



### Library Imports


In [1]:
# import libraries
import ee
import geemap
import numpy as np

### Google Earth Engine Configuration


In [3]:
# Authenticate GEE
ee.Authenticate()

# Initialize GEE
ground_water_project = "groundwater-geo"   

ee.Initialize(project=ground_water_project)

## Study Area  (Kano State)


In [4]:
admin1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

kano = admin1.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Nigeria"),
        ee.Filter.eq("ADM1_NAME", "Kano")
    )
)

kano_geometry = kano.geometry()

### Study Area Map

In [6]:
study_map = geemap.Map()
study_map.center_object(kano, 8)
study_map.add_layer(
    kano.style(color="red", fillColor="00000000", width=2),
    {},
    "Kano Boundary"
)
study_map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

### Reusable Base Maps


In [7]:
def make_kano_map(zoom=8, add_boundary=True):
    m = geemap.Map()
    m.center_object(kano, zoom)
    if add_boundary:
        m.add_layer(
            kano.style(color="red", fillColor="00000000", width=2),
            {},
            "Kano Boundary"
        )
    return m




###  Analysis and Visualization Params


In [10]:
TARGET_CRS = "EPSG:32632"   # WGS 84 / UTM zone 32N
TARGET_SCALE = 30

dem_vis = {
    "min": 300,
    "max": 900,
    "palette": ["0b3d0b", "4f9d4f", "ffff99", "c98b4b", "ffffff"]
}

slope_vis = {
    "min": 0,
    "max": 20,
    "palette": ["ffffff", "ffff00", "ff9900", "ff0000"]
}

landcover_vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400", "#ffbb22", "#ffff4c", "#f096ff",
        "#fa0000", "#b4b4b4", "#f0f0f0", "#0064c8",
        "#0096a0", "#00cf75", "#fae6a0"
    ]
}

soil_vis = {
    "min": 1,
    "max": 12,
    "palette": [
        "#d5c36b", "#b96947", "#9d3706", "#ae868f",
        "#f86714", "#46d143", "#368f20", "#3e5a14",
        "#ffd557", "#fff72e", "#ff5a9d", "#ff005b"
    ]
}

suitability_vis = {
    "min": 1,
    "max": 5,
    "palette": ["red", "orange", "yellow", "lightgreen", "darkgreen"]
}

potential_legend = {
    "Very Low": "red",
    "Low": "orange",
    "Moderate": "yellow",
    "High": "lightgreen",
    "Very High": "darkgreen"
}

##  Thematic Layers

###  Elevation and Slope



In [ ]:
# Elevation and slope are derived from the NASA SRTM 30 m DEM. 

dem = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(kano_geometry)
elevation = dem.rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")

m = make_kano_map(9)
m.add_layer(elevation, dem_vis, "Elevation")
m.add_layer(slope, slope_vis, "Slope")
m

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

###  Land Cover



In [12]:
# ESA WorldCover v200 provides the 2021 land-cover map at 10 m resolution.

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(kano_geometry)
)

m = make_kano_map(9)
m.add_layer(worldcover, landcover_vis, "ESA WorldCover 2021")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

###  Soil Texture



In [13]:
# OpenLandMap USDA soil texture is represented using the topsoil `b0` band.

soil_texture = (
    ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02")
    .select("b0")
    .clip(kano_geometry)
)

m = make_kano_map(9)
m.add_layer(soil_texture, soil_vis, "USDA Soil Texture (0 cm)")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

### Rainfall  (Mean Annual Precipitation)



In [14]:
# CHIRPS daily precipitation is aggregated into annual totals for 2014–2023, and the ten annual totals are averaged to produce mean annual precipitation.

CHIRPS_START_YEAR = 2014
CHIRPS_END_YEAR = 2023

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")

def annual_precipitation(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    return (
        chirps
        .filterDate(start, end)
        .sum()
        .rename("annual_precipitation")
        .set("year", year)
    )

annual_images = [
    annual_precipitation(year)
    for year in range(CHIRPS_START_YEAR, CHIRPS_END_YEAR + 1)
]

annual_precipitation_collection = ee.ImageCollection.fromImages(annual_images)

rainfall = (
    annual_precipitation_collection
    .mean()
    .rename("annual_precipitation")
    .clip(kano_geometry)
)

rainfall_vis = {
    "min": 500,
    "max": 1200,
    "palette": ["f7fbff", "c6dbef", "6baed6", "2171b5", "08306b"]
}

m = make_kano_map(8)
m.add_layer(rainfall, rainfall_vis, "Mean Annual Precipitation (2014–2023)")
m

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

###  Hydrological Layer  (Drainage Density)



In [15]:
# MERIT Hydro provides upstream drainage area at approximately 90 m.

merit_hydro = ee.Image("MERIT/Hydro/v1_0_1")
flow_accumulation = merit_hydro.select("upa").clip(kano_geometry)

STREAM_THRESHOLD_KM2 = 100

stream_mask = flow_accumulation.gte(STREAM_THRESHOLD_KM2).selfMask()

# Approximate drainage density within a 5 km circular neighbourhood.
# MERIT Hydro is approximately 90 m, so each stream pixel is represented
# by an approximate 90 m centreline segment.

window_radius_m = 5000
stream_kernel = ee.Kernel.circle(
    radius=window_radius_m,
    units="meters",
    normalize=False
)

stream_pixel_count = (
    stream_mask
    .unmask(0)
    .reduceNeighborhood(
        reducer=ee.Reducer.sum(),
        kernel=stream_kernel,
        skipMasked=False
    )
)

approx_stream_length_m = stream_pixel_count.multiply(90)
window_area_m2 = 3.141592653589793 * (window_radius_m ** 2)

drainage_density = (
    approx_stream_length_m
    .divide(window_area_m2)
    .multiply(1000)
    .rename("drainage_density")
    .clip(kano_geometry)
)

drainage_density_vis = {
    "min": 0,
    "max": 1,
    "palette": ["ffffff", "9ecae1", "3182bd", "08519c"]
}

m = make_kano_map(9)
m.add_layer(drainage_density, drainage_density_vis, "Drainage Density (km/km²)")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

###  Topographic Wetness Index (TWI)


In [16]:
# TWI is derived as a topographic wetness proxy using MERIT Hydro upstream drainage area and MERIT DEM slope:
# `TWI = ln(flow accumulation / tan(slope))`

merit_dem = ee.Image("MERIT/DEM/v1_0_3").select("dem").clip(kano)
merit_slope = ee.Terrain.slope(merit_dem)

slope_radians = merit_slope.multiply(3.141592653589793 / 180)
tan_slope = slope_radians.tan().max(0.001)

twi = (
    flow_accumulation
    .divide(tan_slope)
    .log()
    .rename("twi")
    .clip(kano)
)

twi_vis = {
    "min": 0,
    "max": 12,
    "palette": [
        "white",
        "yellow",
        "green",
        "blue"
    ]
}

twi_map = make_kano_map(9)

twi_map.addLayer(
    twi,
    twi_vis,
    "Topographic Wetness Index"
)

twi_map

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

##  Reclassification to Groundwater Suitability Scores



In [18]:
# All criteria are converted to a common scale from **1 (least suitable)** to **5 (most suitable)** before weighted overlay.

def percentile_reclassify(image, region, scale, band_name, inverse=False):
    percentiles = image.reduceRegion(
        reducer=ee.Reducer.percentile([20, 40, 60, 80]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e13
    )

    p20 = ee.Number(percentiles.get(f"{band_name}_p20"))
    p40 = ee.Number(percentiles.get(f"{band_name}_p40"))
    p60 = ee.Number(percentiles.get(f"{band_name}_p60"))
    p80 = ee.Number(percentiles.get(f"{band_name}_p80"))

    if not inverse:
        classified = (
            image.lt(p20).multiply(1)
            .add(image.gte(p20).And(image.lt(p40)).multiply(2))
            .add(image.gte(p40).And(image.lt(p60)).multiply(3))
            .add(image.gte(p60).And(image.lt(p80)).multiply(4))
            .add(image.gte(p80).multiply(5))
        )
    else:
        classified = (
            image.gte(p80).multiply(1)
            .add(image.gte(p60).And(image.lt(p80)).multiply(2))
            .add(image.gte(p40).And(image.lt(p60)).multiply(3))
            .add(image.gte(p20).And(image.lt(p40)).multiply(4))
            .add(image.lt(p20).multiply(5))
        )

    return classified.rename(f"{band_name}_suitability")

###  Rainfall Suitability


In [19]:
# Higher mean annual precipitation is assigned higher suitability.

rainfall_suitability = percentile_reclassify(
    rainfall,
    kano_geometry,
    5566,
    "annual_precipitation",
    inverse=False
)

### Elevation Suitability



In [20]:
#  Lower elevations are assigned higher suitability.

elevation_suitability = (
    elevation.expression(
        "(b('elevation') <= 200) ? 5"
        ": (b('elevation') <= 400) ? 4"
        ": (b('elevation') <= 700) ? 3"
        ": (b('elevation') <= 1000) ? 2"
        ": 1"
    )
    .rename("elevation_suitability")
)

### Slope Suitability



In [21]:
# Gentler slopes are assigned higher suitability because they generally favour infiltration relative to steep slopes.

slope_suitability = (
    slope.expression(
        "(b('slope') <= 2) ? 5"
        ": (b('slope') <= 5) ? 4"
        ": (b('slope') <= 10) ? 3"
        ": (b('slope') <= 20) ? 2"
        ": 1"
    )
    .rename("slope_suitability")
)

### Drainage Density Suitability



In [22]:
# Lower drainage density is assigned higher groundwater suitability in the MCDA because high drainage density generally indicates
#  greater surface runoff and comparatively less infiltration opportunity.

drainage_suitability = percentile_reclassify(
    drainage_density,
    kano_geometry,
    500,
    "drainage_density",
    inverse=True
)

### TWI Suitability



In [23]:
# Higher TWI values indicate greater relative wetness/moisture accumulation and are assigned higher suitability.

twi_suitability = (
    twi.expression(
        "(b('twi') <= 2) ? 1"
        ": (b('twi') <= 4) ? 2"
        ": (b('twi') <= 6) ? 3"
        ": (b('twi') <= 8) ? 4"
        ": 5"
    )
    .rename("twi_suitability")
    .clip(kano)
)

### Land Cover Suitability



In [24]:
# Vegetated and cultivated classes receive higher scores.
# Built-up receive lower scores.

landcover_suitability = worldcover.remap(
    [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
    [5, 4, 4, 5, 1, 2, 1, 1, 4, 5, 2]
).rename("landcover_suitability")

###  Soil Texture Suitability


In [25]:
soil_suitability = soil_texture.remap(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    [5, 5, 4, 4, 3, 3, 2, 2, 2, 1, 1, 1]
).rename("soil_suitability")

## Suitability Layer Review



In [27]:
# The seven standardized criteria are assembled into one dictionary for consistent use in visualization, validation and MCDA.

suitability_layers = {
    "Rainfall": rainfall_suitability,
    "Elevation": elevation_suitability,
    "Slope": slope_suitability,
    "Land Cover": landcover_suitability,
    "Soil Texture": soil_suitability,
    "Drainage Density": drainage_suitability,
    "TWI": twi_suitability
}

In [29]:
twi_suitability_map = make_kano_map(9)

twi_suitability_map.addLayer(
    twi_suitability,
    {
        "min": 1,
        "max": 5,
        "palette": [
            "red",
            "orange",
            "yellow",
            "lightgreen",
            "darkgreen"
        ]
    },
    "TWI Suitability"
)

twi_suitability_map.add_legend(
    title="Suitability Score",
    legend_dict={
        "1 — Very Low": "red",
        "2 — Low": "orange",
        "3 — Moderate": "yellow",
        "4 — High": "lightgreen",
        "5 — Very High": "darkgreen"
    }
)

twi_suitability_map

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

## GIS-Based Multi-Criteria Decision Analysis (MCDA)

### Analytic Hierarchy Process (AHP) Weighting

The relative importance of the seven groundwater-potential criteria was determined using the Analytic Hierarchy Process (AHP).

The criteria were compared pairwise using Saaty's fundamental 1–9 comparison scale. A value of 1 indicates equal importance between two criteria, while higher values indicate increasing relative importance of one criterion over another. Reciprocal values are used where the criterion in the row is less important than the criterion in the column.

The pairwise comparison matrix was used to calculate normalized criterion weights. The consistency of the expert judgments was evaluated using the Consistency Index (CI) and Consistency Ratio (CR). A Consistency Ratio below 0.10 was considered acceptable.

The seven criteria considered were rainfall, elevation, slope, land cover, soil texture, drainage density, and Topographic Wetness Index (TWI).

In [ ]:
# AHP criteria
criteria = [
    "Rainfall",
    "Elevation",
    "Slope",
    "Land Cover",
    "Soil Texture",
    "Drainage Density",
    "TWI"
]

# Pairwise comparison matrix
# Rows and columns follow the same order as the criteria above.
# Saaty's 1–9 scale is used, with reciprocal values for opposite comparisons.

ahp_matrix = np.array([
    [1,   3,   3,   2,   1,   2,   1],
    [1/3, 1,   2,   1/2, 1/3, 1/2, 1/3],
    [1/3, 1/2, 1,   1/2, 1/3, 1/2, 1/2],
    [1/2, 2,   2,   1,   1/2, 1,   1/2],
    [1,   3,   3,   2,   1,   2,   1],
    [1/2, 2,   2,   1,   1/2, 1,   1/2],
    [1,   3,   2,   2,   1,   2,   1]
], dtype=float)

# Check matrix dimensions
assert ahp_matrix.shape == (7, 7)

# Check reciprocal property
assert np.allclose(
    ahp_matrix * ahp_matrix.T,
    np.ones((7, 7)),
    atol=1e-9
)

# Calculate principal eigenvector
eigenvalues, eigenvectors = np.linalg.eig(ahp_matrix)

max_index = np.argmax(eigenvalues.real)
lambda_max = eigenvalues[max_index].real

weights = np.abs(eigenvectors[:, max_index].real)
weights = weights / weights.sum()

# Consistency Index
n = len(criteria)
CI = (lambda_max - n) / (n - 1)

# Random Index for n = 7
RI = 1.32

# Consistency Ratio
CR = CI / RI

# Display results
# print("AHP Criterion Weights")
# print("-" * 40)

# for criterion, weight in zip(criteria, weights):
#     print(f"{criterion:<20} {weight:.4f} ({weight * 100:.2f}%)")

# print("\nSum of weights:", weights.sum())
# print("Principal eigenvalue (λmax):", round(lambda_max, 4))
# print("Consistency Index (CI):", round(CI, 4))
# print("Consistency Ratio (CR):", round(CR, 4))

# if CR < 0.10:
# #    print("\nAHP consistency check: ACCEPTABLE (CR < 0.10)")
# else:
#     print("\nAHP consistency check: NOT ACCEPTABLE (CR >= 0.10)")

AHP Criterion Weights
----------------------------------------
Rainfall             0.2125 (21.25%)
Elevation            0.0745 (7.45%)
Slope                0.0649 (6.49%)
Land Cover           0.1161 (11.61%)
Soil Texture         0.2125 (21.25%)
Drainage Density     0.1161 (11.61%)
TWI                  0.2034 (20.34%)

Sum of weights: 1.0
Principal eigenvalue (λmax): 7.1102
Consistency Index (CI): 0.0184
Consistency Ratio (CR): 0.0139

AHP consistency check: ACCEPTABLE (CR < 0.10)


"Criterion weights were derived using AHP through pairwise comparison of the seven selected groundwater-potential factors. The resulting comparison matrix produced a consistency ratio of 0.0139, which is below the accepted threshold of 0.10."

### AHP-Derived Criterion Weights

The AHP procedure produced normalized weights for the seven groundwater-potential criteria. Rainfall and soil texture received the highest weights at 21.25% each, followed by TWI at 20.34%. Land cover and drainage density each received 11.61%, while elevation and slope received 7.45% and 6.49%, respectively.

The weights sum to 1.00 and the calculated Consistency Ratio is 0.0139, indicating an acceptable level of consistency in the pairwise comparisons.

In [ ]:
# Convert the AHP weights into a dictionary

ahp_weights = dict(zip(criteria, weights))

# Confirm that every suitability layer has an AHP weight
assert set(suitability_layers.keys()) == set(ahp_weights.keys())

# Display the final weights used in the MCDA
print("Final AHP Weights Used in MCDA")
print("-" * 40)

for criterion, weight in ahp_weights.items():
    print(f"{criterion:<20} {weight:.4f} ({weight * 100:.2f}%)")

# print("\nWeight total:", sum(ahp_weights.values()))
# print("Consistency Ratio:", round(CR, 4))

Final AHP Weights Used in MCDA
----------------------------------------
Rainfall             0.2125 (21.25%)
Elevation            0.0745 (7.45%)
Slope                0.0649 (6.49%)
Land Cover           0.1161 (11.61%)
Soil Texture         0.2125 (21.25%)
Drainage Density     0.1161 (11.61%)
TWI                  0.2034 (20.34%)

Weight total: 1.0
Consistency Ratio: 0.0139


## Weighted Groundwater Potential Index

The standardized suitability layers were combined using a weighted linear combination based on the AHP-derived criterion weights.

For each location, the groundwater potential index is calculated by multiplying the suitability score of each criterion by its corresponding AHP weight and summing the resulting weighted scores.

The resulting index represents the relative groundwater potential across Kano State. Because the suitability scores range from 1 to 5 and the AHP weights sum to 1.00, the theoretical range of the weighted index is also 1 to 5.

In [37]:
# Weighted Linear Combination
# Calculate the Groundwater Potential Index (GWPI)

groundwater_index = ee.Image(0)

for criterion, weight in ahp_weights.items():
    groundwater_index = groundwater_index.add(
        suitability_layers[criterion].multiply(weight)
    )

groundwater_index = (
    groundwater_index
    .clip(kano)
    .rename("GWPI")
)

# print("Groundwater Potential Index created successfully.")

In [ ]:
# Calculate the GWPI range using a coarser scale for diagnostics.
# This is only for inspecting the index range, not for producing the final map.

gwpi_stats = groundwater_index.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=kano.geometry(),
    scale=1000,
    bestEffort=True,
    maxPixels=1e8
)

# print("GWPI range:")
# print(gwpi_stats.getInfo())

GWPI range:
{'GWPI_max': 4.17416757904275, 'GWPI_min': 1.7373345142481436}


## Groundwater Potential Classification

The Groundwater Potential Index (GWPI) was classified into five groundwater potential zones: Very Low, Low, Moderate, High, and Very High.

The observed GWPI values across Kano State ranged from approximately 1.74 to 4.17. Equal-interval classification was applied across this observed range to divide the index into five potential classes.

The resulting classes represent relative groundwater potential within the study area and should be interpreted as groundwater-potential zones rather than direct measurements of aquifer productivity.

In [40]:
# Observed GWPI range
GWPI_MIN = 1.7373345142481436
GWPI_MAX = 4.17416757904275

# Calculate equal interval
GWPI_INTERVAL = (GWPI_MAX - GWPI_MIN) / 5

# Classification thresholds
class_2 = GWPI_MIN + GWPI_INTERVAL
class_3 = GWPI_MIN + (2 * GWPI_INTERVAL)
class_4 = GWPI_MIN + (3 * GWPI_INTERVAL)
class_5 = GWPI_MIN + (4 * GWPI_INTERVAL)

# print("GWPI Classification Thresholds")
# print("-" * 40)
# print(f"Very Low:   {GWPI_MIN:.4f} – {class_2:.4f}")
# print(f"Low:        {class_2:.4f} – {class_3:.4f}")
# print(f"Moderate:   {class_3:.4f} – {class_4:.4f}")
# print(f"High:       {class_4:.4f} – {class_5:.4f}")
# print(f"Very High:  {class_5:.4f} – {GWPI_MAX:.4f}")

In [41]:
# Classify the Groundwater Potential Index into five zones

groundwater_potential = (
    ee.Image(1)
    .where(groundwater_index.gte(class_2), 2)
    .where(groundwater_index.gte(class_3), 3)
    .where(groundwater_index.gte(class_4), 4)
    .where(groundwater_index.gte(class_5), 5)
    .rename("Groundwater_Potential")
    .clip(kano)
)

# print("Groundwater potential classification created successfully.")

In [42]:
potential_vis = {
    "min": 1,
    "max": 5,
    "palette": [
        "red",
        "orange",
        "yellow",
        "lightgreen",
        "darkgreen"
    ]
}

potential_map = make_kano_map(8)

potential_map.add_layer(
    groundwater_potential,
    potential_vis,
    "Groundwater Potential Zones"
)

potential_map.add_legend(
    title="Groundwater Potential",
    legend_dict={
        "1 — Very Low": "red",
        "2 — Low": "orange",
        "3 — Moderate": "yellow",
        "4 — High": "lightgreen",
        "5 — Very High": "darkgreen"
    }
)

potential_map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

## Priority Areas for Groundwater Development

Areas classified as High and Very High groundwater potential are identified as priority zones for further groundwater investigation and development.

These areas represent relatively favourable groundwater-potential conditions based on the selected criteria and AHP-weighted MCDA model. They should not be interpreted as guaranteed productive aquifer locations; site-specific hydrogeological investigations are recommended before groundwater development decisions.

In [43]:
# Identify High and Very High groundwater-potential zones
priority_areas = groundwater_potential.gte(4).selfMask()

priority_map = make_kano_map(8)

priority_map.addLayer(
    priority_areas,
    {
        "min": 4,
        "max": 5,
        "palette": ["darkgreen"]
    },
    "Priority Groundwater Development Areas"
)

priority_map.add_legend(
    title="Priority Zone",
    legend_dict={
        "High / Very High Potential": "darkgreen"
    }
)

priority_map

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

In [44]:
# Calculate area of High + Very High groundwater-potential zones

priority_area_stats = (
    ee.Image.pixelArea()
    .updateMask(priority_areas)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=kano.geometry(),
        scale=1000,
        bestEffort=True,
        maxPixels=1e8
    )
)

priority_area_km2 = ee.Number(
    priority_area_stats.get("area")
).divide(1e6)

print(
    "Priority groundwater-potential area (km²):",
    priority_area_km2.getInfo()
)

Priority groundwater-potential area (km²): 3888.4839804740204


In [ ]:
# Calculate total Kano study-area size

total_area_stats = (
    ee.Image.pixelArea()
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=kano.geometry(),
        scale=1000,
        bestEffort=True,
        maxPixels=1e8
    )
)

total_area_km2 = ee.Number(
    total_area_stats.get("area")
).divide(1e6)

priority_percentage = (
    priority_area_km2
    .divide(total_area_km2)
    .multiply(100)
)

# print("Total study area (km²):", total_area_km2.getInfo())
# print("Priority area (km²):", priority_area_km2.getInfo())
# print("Priority area (%):", priority_percentage.getInfo())

Total study area (km²): 20082.20634142575
Priority area (km²): 3888.4839804740204
Priority area (%): 19.362832521309283


### Priority Area Statistics

The MCDA identified approximately 3,888.48 km² of the study area as High or Very High groundwater-potential zones. This represents approximately 19.36% of the total Kano study area of 20,082.21 km².

## Results

The GIS-based Multi-Criteria Decision Analysis (MCDA) produced a continuous Groundwater Potential Index (GWPI) and a classified groundwater-potential map for Kano State. The model integrated seven standardized criteria: rainfall, elevation, slope, land cover, soil texture, drainage density, and Topographic Wetness Index (TWI), using AHP-derived criterion weights.

The AHP analysis produced a Consistency Ratio (CR) of 0.0139, which is below the commonly accepted threshold of 0.10. This indicates that the pairwise comparisons used to derive the criterion weights were sufficiently consistent.

The resulting GWPI values ranged from approximately 1.74 to 4.17 across the study area. The index was classified into five relative groundwater-potential categories: Very Low, Low, Moderate, High, and Very High using equal-interval classification across the observed GWPI range.

The analysis identified approximately 3,888.48 km² as High or Very High groundwater-potential areas. This represents approximately 19.36% of the total study area of 20,082.21 km².

The High and Very High potential zones were subsequently extracted as priority areas for further groundwater investigation and development. These areas represent locations where the combination of the selected environmental, topographic, hydrological, soil, and land-cover conditions produced relatively favourable groundwater-potential scores within the MCDA framework.

## Limitations

Several limitations should be considered when interpreting the groundwater-potential results.

### 1. Dependence on the selected criteria

The groundwater-potential model is based on seven selected criteria: rainfall, elevation, slope, land cover, soil texture, drainage density, and Topographic Wetness Index. Groundwater occurrence is influenced by additional hydrogeological factors, including geology, aquifer characteristics, fracture and lineament distribution, groundwater levels, hydraulic properties, and recharge conditions. These factors were not explicitly incorporated into the present MCDA.

### 2. Subjectivity of AHP weighting

Although the AHP pairwise comparison matrix produced a low Consistency Ratio (CR = 0.0139), the resulting weights remain dependent on the assumptions and judgments used to construct the comparison matrix. A consistent matrix does not necessarily mean that the assigned relative importance of the criteria is objectively optimal.

### 3. Linear combination of criteria

The weighted linear combination assumes that the contribution of each criterion can be represented through a standardized suitability score and an additive weight. In reality, groundwater systems involve complex interactions between environmental and hydrogeological factors that may be nonlinear. The MCDA therefore provides a spatial decision-support framework rather than a complete physical simulation of groundwater occurrence.

### 4. Spatial resolution and data uncertainty

The reliability of the final groundwater-potential map depends on the spatial resolution, quality, temporal characteristics, and classification accuracy of the input datasets. Differences in spatial resolution between datasets may introduce uncertainty when the criteria are combined into a common analysis.

### 5. Reclassification assumptions

Several criteria were converted to five suitability classes using predefined thresholds or percentile-based classification. These thresholds simplify continuous environmental conditions into discrete suitability scores and may influence the resulting spatial pattern.

### 6. Lack of direct hydrogeological validation

The groundwater-potential zones have not been validated against an independent dataset of borehole yields, groundwater levels, pumping-test results, geophysical surveys, or other direct measurements of aquifer productivity. Consequently, the High and Very High zones should be interpreted as areas of relatively favourable groundwater potential rather than confirmed productive aquifer locations.

### 7. Interpretation of priority areas

The identified priority areas are intended to support screening and prioritisation of locations for further investigation. They should not be interpreted as locations where groundwater extraction is automatically suitable or guaranteed to be successful. Site-specific hydrogeological investigation remains necessary before drilling or groundwater-development decisions are made.

## Conclusion and Recommendations

### Conclusion

This study developed a GIS-based groundwater-potential assessment for Kano State using Multi-Criteria Decision Analysis (MCDA) and the Analytic Hierarchy Process (AHP). Seven groundwater-controlling criteria—rainfall, elevation, slope, land cover, soil texture, drainage density, and Topographic Wetness Index—were standardized to a common suitability scale and integrated using AHP-derived weights.

The AHP weighting produced a Consistency Ratio of 0.0139, indicating an acceptable level of consistency in the pairwise comparison matrix. The resulting Groundwater Potential Index ranged from approximately 1.74 to 4.17 and was classified into five relative groundwater-potential zones.

The analysis identified approximately 3,888.48 km², equivalent to 19.36% of the 20,082.21 km² study area, as High or Very High groundwater-potential zones. These areas provide a spatial basis for prioritising locations for further groundwater investigation.

Overall, the resulting groundwater-potential map demonstrates the usefulness of GIS-based MCDA for integrating multiple spatial factors into a single decision-support framework. However, the results represent relative groundwater potential and should be treated as a screening and prioritisation product rather than a direct prediction of groundwater yield.

### Recommendations

1. **Prioritise field investigation:** High and Very High groundwater-potential areas should receive priority for detailed hydrogeological investigation and field verification.

2. **Integrate hydrogeological data:** Future assessments should incorporate geology, aquifer characteristics, groundwater levels, borehole yields, geophysical surveys, and other available hydrogeological information.

3. **Validate the model:** The groundwater-potential zones should be evaluated against independent borehole, pumping-test, groundwater-level, or geophysical data to assess the predictive reliability of the model.

4. **Improve spatial data:** Higher-resolution and locally validated datasets should be incorporated where available to reduce uncertainty in the thematic layers.

5. **Conduct sensitivity analysis:** Future work should examine how changes in criterion weights and suitability thresholds affect the spatial distribution of groundwater-potential zones.

6. **Support sustainable development:** Priority zones should be considered within a broader groundwater-management framework that accounts for groundwater demand, recharge, abstraction pressure, and long-term resource sustainability.

7. **Use the map as a screening tool:** The final groundwater-potential map should guide the selection of locations for further investigation rather than serve as a standalone basis for borehole siting or groundwater-development decisions.

### Helper Functions

#### FeatureCollection export function

In [46]:
# Export a FeatureCollection to Google Drive

def export_feature_collection_to_drive(
    feature_collection,
    description,
    folder,
    file_name_prefix,
    file_format="SHP"
):
    task = ee.batch.Export.table.toDrive(
        collection=feature_collection,
        description=description,
        folder=folder,
        fileNamePrefix=file_name_prefix,
        fileFormat=file_format
    )

    task.start()

    print(f"Export task started: {description}")
    
    return task

### Raster/Image export function

In [47]:
# Export an Earth Engine image to Google Drive

def export_image_to_drive(
    image,
    description,
    folder,
    file_name_prefix,
    region,
    scale=30,
    crs="EPSG:4326",
    file_format="GeoTIFF",
    max_pixels=1e13
):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=file_name_prefix,
        region=region,
        scale=scale,
        crs=crs,
        maxPixels=max_pixels,
        fileFormat=file_format
    )

    task.start()

    print(f"Export task started: {description}")
    
    return task

# Exports



### Study-Area Boundary

In [57]:
# Export Kano State study-area boundary

kano_boundary_task = export_feature_collection_to_drive(
    feature_collection=kano,
    description="Kano_Study_Area_Boundary",
    folder="Kano_Groundwater_Boundary",
    file_name_prefix="Kano_Study_Area_Boundary",
    file_format="SHP"
)

Export task started: Kano_Study_Area_Boundary


### Thematic Layers

The thematic datasets used in the analysis 

In [ ]:
# Export original thematic layers

thematic_layers = {
    "Rainfall": rainfall,
    "Elevation": elevation,
    "Slope": slope,
    "Landcover": worldcover,
    "Soil_Texture": soil_texture,
    "Drainage_Density": drainage_density,
    "TWI": twi
}

thematic_tasks = {}

for name, image in thematic_layers.items():

    task = export_image_to_drive(
        image=image,
        description=f"Kano_Thematic_{name}",
        folder="Kano_Groundwater_Thematic",
        file_name_prefix=f"Kano_Thematic_{name}",
        region=kano.geometry(),
        scale=TARGET_SCALE,
        crs=TARGET_CRS
    )

    thematic_tasks[name] = task

# print("All thematic-layer export tasks have been started.")

Export task started: Kano_Thematic_Rainfall
Export task started: Kano_Thematic_Elevation
Export task started: Kano_Thematic_Slope
Export task started: Kano_Thematic_Landcover
Export task started: Kano_Thematic_Soil_Texture
Export task started: Kano_Thematic_Drainage_Density
Export task started: Kano_Thematic_TWI
All thematic-layer export tasks have been started.


### Suitability Layers
Reclassified Suitability Layers

The thematic layers are reclassified to a common groundwater-suitability scale from 1 (Very Low) to 5 (Very High).

In [ ]:
# Export reclassified suitability layers

suitability_layers = {
    "Rainfall": rainfall_suitability,
    "Elevation": elevation_suitability,
    "Slope": slope_suitability,
    "Landcover": landcover_suitability,
    "Soil_Texture": soil_suitability,
    "Drainage_Density": drainage_suitability,
    "TWI": twi_suitability
}

suitability_tasks = {}

for name, image in suitability_layers.items():

    task = export_image_to_drive(
        image=image,
        description=f"Kano_Suitability_{name}",
        folder="Kano_Groundwater_Suitability",
        file_name_prefix=f"Kano_Suitability_{name}",
        region=kano.geometry(),
        scale=TARGET_SCALE,
        crs=TARGET_CRS
    )

    suitability_tasks[name] = task

# print("All suitability-layer export tasks have been started.")

Export task started: Kano_Suitability_Rainfall
Export task started: Kano_Suitability_Elevation
Export task started: Kano_Suitability_Slope
Export task started: Kano_Suitability_Landcover
Export task started: Kano_Suitability_Soil_Texture
Export task started: Kano_Suitability_Drainage_Density
Export task started: Kano_Suitability_TWI
All suitability-layer export tasks have been started.


### Final Outputs
Final Groundwater Potential outputs of the MCDA

In [ ]:
# Export final groundwater-potential outputs

final_outputs = {
    "GWPI": groundwater_index,
    "Groundwater_Potential_Zones": groundwater_potential,
    "Priority_Groundwater_Areas": priority_areas
}

final_tasks = {}

for name, image in final_outputs.items():

    task = export_image_to_drive(
        image=image,
        description=f"Kano_Final_{name}",
        folder="Kano_Groundwater_Final",
        file_name_prefix=f"Kano_Final_{name}",
        region=kano.geometry(),
        scale=TARGET_SCALE,
        crs=TARGET_CRS
    )

    final_tasks[name] = task

# print("All final groundwater-potential export tasks have been started.")

Export task started: Kano_Final_GWPI
Export task started: Kano_Final_Groundwater_Potential_Zones
Export task started: Kano_Final_Priority_Groundwater_Areas
All final groundwater-potential export tasks have been started.
